In [ ]:
from pyspark.sql import SparkSession

spark=SparkSession.builder \
.appName("Working with Joins") \
.getOrCreate()

In [ ]:
%%writefile patients.csv
patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active

In [ ]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consult,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2500,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

In [ ]:
#1
patients_df = spark.read.option(
    "header",
    True
).csv(
    "patients.csv"
)

patients_df.show()

In [ ]:
#2
appointments_df = spark.read.option(
    "header",
    True
).csv(
    "appointments.csv"
)

appointments_df.show()

In [ ]:
#3
patients_df.printSchema()

In [ ]:
#4
patients_df.count()

In [ ]:
#5
patients_df.show(
    5
)

In [ ]:
#6
patients_df.select(
    "city"
).distinct().show()

In [ ]:
#7
appointments_df.select(
    "department"
).distinct().show()

In [ ]:
#8
patients_df.write.mode(
    "overwrite"
).parquet(
    "patients_parquet"
)

In [ ]:
#9
parquet_df = spark.read.parquet(
    "patients_parquet"
)
parquet_df.show()

In [ ]:
#10
print(
    "CSV Count:",
    patients_df.count()
)

print(
    "Parquet Count:",
    parquet_df.count()
)

In [ ]:
from pyspark.sql.functions import *

In [ ]:
#11
patients_df.filter(
    col("city") == "Hyderabad"
).show()

In [ ]:
#12
patients_df.filter(
    col("gender") == "Female"
).show()

In [ ]:
#13
patients_df.filter(
    col("age") > 40
).show()

In [ ]:
#14
appointments_df.filter(
    col("status") == "Completed"
).show()

In [ ]:
#15
appointments_df.filter(
    col("status") == "Pending"
).show()

In [ ]:
#16
appointments_df.filter(
    col("consult") > 1500
).show()

In [ ]:
#17
patients_df.filter(
    col("insurance_status") == "Active"
).show()

In [ ]:
#18
patients_df.filter(
    col("insurance_status") == "Inactive"
).show()

In [ ]:
#19
patients_df.filter(
    col("blood_group") == "O+"
).show()

In [ ]:
#20
appointments_df.filter(
    col("department") == "Cardiology"
).show()

In [ ]:
#21
patients_df.filter(
    col("city").isNull()
).show()

In [ ]:
#22
patients_df.filter(
    col("blood_group").isNull()
).show()

In [ ]:
#23
appointments_df.filter(
    col("consult").isNull()
).show()

In [ ]:
#24
from pyspark.sql.functions import *

patients_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in patients_df.columns
]).show()

In [ ]:
#25
patients_df.fillna(
    {
        "city":"Unknown"
    }
).show()

In [ ]:
#26
patients_df.fillna(
    {
        "blood_group":"Not Available"
    }
).show()

In [ ]:
#27
appointments_df.fillna(
    {
        "consult":0
    }
).show()

In [ ]:
#28
appointments_df.na.drop(
    subset=["consult"]
).show()

In [ ]:
#29
from pyspark.sql.functions import when

patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull(),
        "Incomplete"
    ).otherwise(
        "Complete"
    )
).show()

In [ ]:
#30
patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull(),
        "Incomplete"
    ).otherwise(
        "Complete"
    )
).groupBy(
    "data_quality_status"
).count().show()

In [ ]:
#31
patients_df.withColumn(
    "patient_name",
    upper(col("patient_name"))
).show()

In [ ]:
#32
patients_df.withColumn(
    "patient_name",
    lower(col("patient_name"))
).show()

In [ ]:
#33
patients_df.withColumn(
    "name_length",
    length(col("patient_name"))
).show()

In [ ]:
#34
patients_df.withColumn(
    "first_three_letters",
    substring(col("patient_name"),1,3)
).show()

In [ ]:
#35
patients_df.withColumn(
    "age_group",
    when(col("age") < 30,"Young")
    .when(col("age") < 50,"Adult")
    .otherwise("Senior")
).show()

In [ ]:
#36
patients_df.withColumn(
    "insurance_flag",
    when(
        col("insurance_status") == "Active",
        "Yes"
    ).otherwise(
        "No"
    )
).show()

In [ ]:
#37
patients_df.withColumn(
    "senior_citizen",
    when(
        col("age") >= 60,
        "Yes"
    ).otherwise(
        "No"
    )
).show()

In [ ]:
#38
patients_df.withColumn(
    "name_city",
    concat_ws(
        " ",
        col("patient_name"),
        col("city")
    )
).show()

In [ ]:
#39
patients_df.withColumn(
    "patient_name",
    trim(col("patient_name"))
).show()

In [ ]:
#40
patients_df.withColumn(
    "city",
    upper(col("city"))
).show()

In [ ]:
#41
patients_df.groupBy(
    "city"
).count().show()

In [ ]:
#42
patients_df.groupBy(
    "gender"
).count().show()

In [ ]:
#43
patients_df.groupBy(
    "blood_group"
).count().show()

In [ ]:
#44
appointments_df.groupBy(
    "department"
).count().show()

In [ ]:
#45
patients_df.groupBy(
    "city"
).agg(
    avg("age").alias("average_age")
).show()

In [ ]:
#46
patients_df.groupBy(
    "city"
).agg(
    max("age").alias("maximum_age")
).show()

In [ ]:
#47
patients_df.groupBy(
    "city"
).agg(
    min("age").alias("minimum_age")
).show()

In [ ]:
#48
appointments_df.groupBy(
    "department"
).agg(
    avg(
        col("consult").cast("int")
    ).alias("average_fee")
).show()

In [ ]:
#49
appointments_df.groupBy(
    "department"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_fee")
).show()

In [ ]:
#50
appointments_df.groupBy(
    "department"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
).show(1)

In [ ]:
#51
patients_df.join(
    appointments_df,
    "patient_id",
    "inner"
).show()

In [ ]:
#52
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).show()

In [ ]:
#53
patients_df.join(
    appointments_df,
    "patient_id",
    "right"
).show()

In [ ]:
#54
patients_df.join(
    appointments_df,
    "patient_id",
    "full"
).show()

In [ ]:
#55
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).filter(
    appointments_df.patient_id.isNull()
).show()

In [ ]:
#56
appointments_df.join(
    patients_df,
    "patient_id",
    "left"
).filter(
    patients_df.patient_id.isNull()
).show()

In [ ]:
#57
appointments_df.groupBy(
    "patient_id"
).count().show()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
#58
appointments_df.groupBy(
    "patient_id"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_fees")
).show()

In [ ]:
#59
appointments_df.groupBy(
    "patient_id"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_fees")
).orderBy(
    col("total_fees").desc()
).show(1)

In [ ]:
#60
appointments_df.groupBy(
    "patient_id"
).count().withColumnRenamed(
    "count",
    "appointment_count"
).show()

In [ ]:
#61
from pyspark.sql.functions import *
from pyspark.sql.window import Window

fees_df = appointments_df.groupBy(
    "patient_id"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_fees")
)

In [ ]:
#61
window_spec = Window.orderBy(
    col("total_fees").desc()
)

fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

In [ ]:
#62
window_spec = Window.orderBy(
    col("total_fees").desc()
)

fees_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

In [ ]:
#63
window_spec = Window.orderBy(
    col("total_fees").desc()
)

fees_df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

In [ ]:
#64
window_spec = Window.orderBy(
    col("total_fees").desc()
)

fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

In [ ]:
#65
window_spec = Window.orderBy(
    col("total_fees").desc()
)

fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") <= 3
).show()

In [ ]:
city_fees_df = patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "city",
    "patient_id"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_fees")
)

In [ ]:
#66
window_spec = Window.partitionBy(
    "city"
).orderBy(
    col("total_fees").desc()
)

city_fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

In [ ]:
#67
window_spec = Window.partitionBy(
    "city"
).orderBy(
    col("total_fees")
)

city_fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

In [ ]:
#68
window_spec = Window.orderBy(
    "patient_id"
)

fees_df.withColumn(
    "running_total",
    sum("total_fees").over(window_spec)
).show()

In [ ]:
#69
window_spec = Window.orderBy(
    col("total_fees")
)

fees_df.withColumn(
    "next_fee",
    lead("total_fees").over(window_spec)
).show()

In [ ]:
#70
window_spec = Window.orderBy(
    col("total_fees")
)

fees_df.withColumn(
    "previous_fee",
    lag("total_fees").over(window_spec)
).show()

In [ ]:
%%writefile patient_preferences.json
[
{
"patient_id":101,
"preferred_hospital":"Apollo",
"contact":{
"phone":"9876500011",
"email":"rahul@gmail.com"
}
},
{
"patient_id":102,
"preferred_hospital":"Yashoda",
"contact":{
"phone":null,
"email":"priya@gmail.com"
}
},
{
"patient_id":103,
"preferred_hospital":"Care",
"contact":{
"phone":"9876500013",
"email":null
}
},
{
"patient_id":104,
"preferred_hospital":null,
"contact":{
"phone":"9876500014",
"email":"sneha@gmail.com"
}

}
]

In [ ]:
#71
preferences_df = spark.read.option(
    "multiline",
    True
).json(
    "patient_preferences.json"
)

preferences_df.show()

In [ ]:
#72
preferences_df.printSchema()

In [ ]:
#73
preferences_df.select(
    "patient_id",
    "contact.phone"
).show()

In [ ]:
#74
preferences_df.select(
    "patient_id",
    "contact.email"
).show()

In [ ]:
#75
preferences_df.filter(
    col("contact.phone").isNull()
).show()

In [ ]:
#76
preferences_df.filter(
    col("contact.email").isNull()
).show()

In [ ]:
#77
preferences_df.filter(
    col("preferred_hospital").isNull()
).show()

In [ ]:
#78
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    when(
        col("contact.phone").isNull(),
        "Not Available"
    ).otherwise(
        col("contact.phone")
    ).alias("phone")
).show()

In [ ]:
#79
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    when(
        col("contact.email").isNull(),
        "Not Available"
    ).otherwise(
        col("contact.email")
    ).alias("email")
).show()

In [ ]:
#80
patients_df.join(
    preferences_df,
    "patient_id",
    "inner"
).show()

In [ ]:
#81
patients_df.createOrReplaceTempView(
    "patients"
)

In [ ]:
#82
appointments_df.createOrReplaceTempView(
    "appointments"
)

In [ ]:
#83
spark.sql(
    "select * from patients"
).show()

In [ ]:
#84
spark.sql(
    """
    select *
    from patients
    where city='Hyderabad'
    """
).show()

In [ ]:
#85
spark.sql(
    """
    select city,
           count(*) as total_patients
    from patients
    group by city
    """
).show()

In [ ]:
#86
spark.sql(
    """
    select department,
           count(*) as total_appointments
    from appointments
    group by department
    """
).show()

In [ ]:
#87
spark.sql(
    """
    select department,
           avg(cast(consult as int)) as average_fee
    from appointments
    group by department
    """
).show()

In [ ]:
#88
spark.sql(
    """
    select max(cast(consult as int)) as highest_fee
    from appointments
    """
).show()

In [ ]:
#89
spark.sql(
    """
    select patient_id,
           count(*) as appointment_count
    from appointments
    group by patient_id
    """
).show()

In [ ]:
#90
spark.sql(
    """
    select patient_id,
           sum(cast(consult as int)) as total_fees
    from appointments
    group by patient_id
    order by total_fees desc
    limit 5
    """
).show()

In [ ]:
#91
patients_df = spark.read.option(
    "header",
    True
).csv(
    "patients.csv"
)

appointments_df = spark.read.option(
    "header",
    True
).csv(
    "appointments.csv"
)

In [ ]:
#92
preferences_df = spark.read.option(
    "multiline",
    True
).json(
    "patient_preferences.json"
)

In [ ]:
#93
patients_df = patients_df.fillna(
    {
        "city":"Unknown",
        "blood_group":"Not Available"
    }
)

appointments_df = appointments_df.fillna(
    {
        "consult":0
    }
)

In [ ]:
#94
final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    preferences_df,
    "patient_id",
    "left"
)

final_df.show()

In [ ]:
#95
final_df = final_df.withColumn(
    "age_group",
    when(col("age") < 30,"Young")
    .when(col("age") < 50,"Adult")
    .otherwise("Senior")
)

final_df.show()

In [ ]:
#96
final_df = final_df.withColumn(
    "revenue",
    col("consult").cast("int")
)

final_df.show()

In [ ]:
#97
final_df.groupBy(
    "patient_id"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_spending")
).show()

In [ ]:
#98
final_df.groupBy(
    "department"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("department_revenue")
).show()

In [ ]:
#99
final_df.write.mode(
    "overwrite"
).parquet(
    "hospital_analytics_output"
)

In [ ]:
#100
print("Hospital Analytics Report")

print("Total Patients")
print(patients_df.count())

print("Total Appointments")
print(appointments_df.count())

print("Department Revenue")
final_df.groupBy(
    "department"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_revenue")
).show()

print("Patient Wise Spending")
final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum(
        col("consult").cast("int")
    ).alias("total_spending")
).show()

print("Appointments By Department")
final_df.groupBy(
    "department"
).count().show()